# Chapter 8: From dice to A/B tests

In [1]:
import numpy as np
from expkit.sim.abtest import two_arm_binary, two_arm_continuous, stratified_binary
from expkit.inference.normal import two_proportion_z, two_sample_t
from expkit.plot.style import apply_style
apply_style()

## Loop A: binary outcome at two sample sizes

In [2]:
for n in [1000, 10000]:
    exp = two_arm_binary(n, 0.05, 0.06, seed=80 + n)
    z = two_proportion_z(int(exp.treatment.sum()), n, int(exp.control.sum()), n)
    rng = np.random.default_rng(0)
    a_c, b_c = 1 + int(exp.control.sum()), 1 + (n - int(exp.control.sum()))
    a_t, b_t = 1 + int(exp.treatment.sum()), 1 + (n - int(exp.treatment.sum()))
    diffs = rng.beta(a_t, b_t, 5000) - rng.beta(a_c, b_c, 5000)
    print(f'N={n:>6}  z={z.statistic:.2f}  p={z.p_value:.4g}  P(diff>0)={(diffs > 0).mean():.3f}')

N=  1000  z=-0.31  p=0.7593  P(diff>0)=0.381
N= 10000  z=3.20  p=0.001363  P(diff>0)=0.999


## Loop B: continuous outcome (revenue)

In [3]:
exp = two_arm_continuous(2000, mu_control=10.0, mu_treatment=10.5, sigma=4.0, seed=82)
t = two_sample_t(exp.treatment, exp.control, equal_var=False)
print(f'mean control = {exp.control.mean():.3f}, treatment = {exp.treatment.mean():.3f}')
print(f'Welch t = {t.statistic:.3f}, p = {t.p_value:.4g}, point estimate = {t.point_estimate:.3f}')

mean control = 10.042, treatment = 10.491
Welch t = 3.561, p = 0.0003732, point estimate = 0.449


## Loop C: stratification

In [4]:
strata = stratified_binary({'power-users': 200, 'casual': 1800}, {'power-users': (0.30, 0.32), 'casual': (0.05, 0.07)}, seed=83)
for name, s in strata.items():
    n = len(s.control)
    z = two_proportion_z(int(s.treatment.sum()), n, int(s.control.sum()), n)
    print(f'{name:<14} N={2*n:>5}  diff={s.treatment.mean()-s.control.mean():+.3f}  p={z.p_value:.4g}')
import numpy as np
c = np.concatenate([s.control for s in strata.values()])
t = np.concatenate([s.treatment for s in strata.values()])
z = two_proportion_z(int(t.sum()), len(t), int(c.sum()), len(c))
print(f'pooled         N={len(t)+len(c):>5}  diff={t.mean()-c.mean():+.3f}  p={z.p_value:.4g}')

power-users    N=  400  diff=+0.055  p=0.2448
casual         N= 3600  diff=+0.024  p=0.0018
pooled         N= 4000  diff=+0.027  p=0.002084
